# Testing Deployment TensorFlow Serving
Notebook ini menguji layanan **TensorFlow Serving yang benar-benar di-deploy ke Railway**. URL di bawah bukan placeholder. Pengujian meliputi status model, metadata signature, dan prediction request. Output HTTP disimpan di notebook sebagai bukti deployment dapat diakses melalui internet.

In [ ]:
import json
import requests

BASE_URL = "https://mlops-dicoding-reza-production.up.railway.app"
MODEL_NAME = "breast_cancer_model"
STATUS_URL = f"{BASE_URL}/v1/models/{MODEL_NAME}"
METADATA_URL = f"{BASE_URL}/v1/models/{MODEL_NAME}/metadata"

status_response = requests.get(STATUS_URL, timeout=60)
print("GET", STATUS_URL)
print("HTTP status:", status_response.status_code)
print(json.dumps(status_response.json(), indent=2))
status_response.raise_for_status()

metadata_response = requests.get(METADATA_URL, timeout=60)
print("\nGET", METADATA_URL)
print("HTTP status:", metadata_response.status_code)
print(json.dumps(metadata_response.json(), indent=2)[:5000])
metadata_response.raise_for_status()

## Prediction request
Endpoint produksi saat ini menerima satu vektor berisi 30 fitur numerik pada `instances`. Data uji diambil langsung dari baris pertama dataset submission dan dikirim ke endpoint `:predict`.

In [ ]:
import pandas as pd

features = (pd.read_csv("data/breast_cancer.csv")
            .drop(columns=["label"])
            .iloc[0]
            .astype("float32")
            .tolist())
payload = {"instances": [features]}
PREDICT_URL = f"{BASE_URL}/v1/models/{MODEL_NAME}:predict"
prediction_response = requests.post(PREDICT_URL, json=payload, timeout=60)
print("POST", PREDICT_URL)
print("HTTP status:", prediction_response.status_code)
print(json.dumps(prediction_response.json(), indent=2))
prediction_response.raise_for_status()